In [6]:
from pybaseball import statcast_single_game, playerid_reverse_lookup
import statsapi as MLB_API
import pandas as pd

## Getting info and things

In [7]:
print("lookup team")
tor = MLB_API.lookup_team("tor")
last_game_id = MLB_API.last_game(tor[0]['id'])
game_info:pd.DataFrame = statcast_single_game(last_game_id)
if not game_info.empty:
    print("got game👍")
else:
    print("get game failed :(")
    raise RuntimeError("failed to get game")

lookup team
got game👍


c:\Python313\Lib\site-packages\pybaseball\datahelpers\postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)


## Game info

In [ ]:
# keys = game_info.keys()

# with open('game_info_keys.txt', 'w') as f:
#     for key in keys:
#         f.write(f"{key}\n")
        
home_team = game_info.iloc[0]['home_team']
away_team = game_info.iloc[0]['away_team']

print(f"Game Date: {game_info.iloc[0]['game_date']}")
print(f"{away_team} @ {home_team}")
print(f"{game_info['post_away_score'].max()} - {game_info['post_home_score'].max()}")
if not game_info.iloc[0]['inning'] == 9:
    print(f"Extra innings - ({game_info.iloc[0]['inning']})")

Game Date: 2026-04-04 00:00:00
TOR @ CWS
3 - 6


## Batter Info

In [9]:
batters = game_info['batter'].unique().tolist()

print(f"Total Unique Batters: {len(batters)}")

names:pd.DataFrame = playerid_reverse_lookup(batters, key_type='mlbam')
batters:tuple[int, str, str] = list(zip(names['key_mlbam'], names['name_first'], names['name_last']))

batter_info = []

for batter in batters:
    b_data = {
        'id': batter[0],
        'first_name': batter[1].capitalize(),
        'last_name': batter[2].capitalize(),
        'total_estimated_woba': float(round(game_info.loc[game_info['batter'] == batter[0], 'estimated_woba_using_speedangle'].sum(), 3)),
        'total_estimated_slg': float(round(game_info.loc[game_info['batter'] == batter[0], 'estimated_slg_using_speedangle'].sum(), 3)),
        'home_team': game_info.loc[game_info['batter'] == batter[0], 'inning_topbot'].iloc[0] == 'Bot'
    }
    batter_info.append(b_data)
    print(f"---{b_data['last_name']}, {b_data['first_name']}---")
    print(f"Team = {home_team if b_data['home_team'] else away_team}")
    print(f"Estimated times reaching base: {b_data['total_estimated_woba']}")
    print(type(b_data['total_estimated_woba']))
    print(f"Estimated total bases: {b_data['total_estimated_slg']}")


Total Unique Batters: 22
---Schneider, Davis---
Team = TOR
Estimated times reaching base: 1.379
<class 'float'>
Estimated total bases: 0.883
---Lukes, Nathan---
Team = TOR
Estimated times reaching base: 0.07
<class 'float'>
Estimated total bases: 0.054
---Barger, Addison---
Team = TOR
Estimated times reaching base: 0.7
<class 'float'>
Estimated total bases: 0.0
---Peters, Tristan---
Team = CWS
Estimated times reaching base: 0.254
<class 'float'>
Estimated total bases: 0.273
---Benintendi, Andrew---
Team = CWS
Estimated times reaching base: 0.428
<class 'float'>
Estimated total bases: 0.508
---Meidroth, Chase---
Team = CWS
Estimated times reaching base: 0.416
<class 'float'>
Estimated total bases: 0.565
---Clement, Ernie---
Team = TOR
Estimated times reaching base: 0.412
<class 'float'>
Estimated total bases: 0.481
---Straw, Myles---
Team = TOR
Estimated times reaching base: 1.015
<class 'float'>
Estimated total bases: 1.244
---Mcguire, Reese---
Team = CWS
Estimated times reaching base:

## Team Splits and Game Predictions

In [10]:
home_info = []
away_info = []
for batter in batter_info:
    if batter['home_team']:
        home_info.append(batter)
    else:
        away_info.append(batter)

def print_team_info(team_info:list, home_away:bool) -> None:
    print(f'-----{home_team if home_away else away_team} Team Info-----')
    print(f'Estimated times reaching base: {sum(batter['total_estimated_woba'] for batter in team_info)}')
    print(f'Estimated bases: {sum(batter['total_estimated_slg'] for batter in team_info)}')
    print('\n')

print_team_info(home_info, True)
print_team_info(away_info, False)

print('-----Game Analysis-----')
winometer = sum(batter['total_estimated_slg'] for batter in home_info) / (sum(batter['total_estimated_slg'] for batter in home_info) + sum(batter['total_estimated_slg'] for batter in away_info))
print(f'{home_team} Deserve-to-Win-O-Meter: {round(winometer, 4) * 100}%')
print(f'{away_team} Deserve-to-Win-O-Meter: {round(1 - winometer, 4) * 100}%')

-----CWS Team Info-----
Estimated times reaching base: 8.759
Estimated bases: 9.442


-----TOR Team Info-----
Estimated times reaching base: 12.636
Estimated bases: 14.759


-----Game Analysis-----
CWS Deserve-to-Win-O-Meter: 39.01%
TOR Deserve-to-Win-O-Meter: 60.99%
